In [ ]:
# OLIST DATA ANALYTICS HACKATHON — COMPLETE ANALYSIS
# Q1: Marketplace Performance Over Time
# Q2: Delivery Performance & Customer Satisfaction
# Q3: Seller & Geographic Patterns
# Q4: Product Category Performance
# Q5: Payment Behavior
# Q6: Root Cause Analysis


!pip -q install duckdb openpyxl

import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import files
from IPython.display import display

print("="*70)
print("OLIST DATA ANALYTICS HACKATHON")
print("="*70)


# 1. UPLOAD CLEANED CSV


print("\nUPLOAD YOUR CLEANED CSV FILE")
print("Example: olist_master_powerbi(1).csv")

uploaded = files.upload()
clean_file = list(uploaded.keys())[0]

print("\nLoaded:", clean_file)


# 2. LOAD CLEANED DATA


df = pd.read_csv(clean_file)

print("\nDataset shape:", df.shape)

# Convert dates
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Convert numeric columns
numeric_cols = [
    'delivery_delay_days',
    'total_payment_value',
    'total_item_price',
    'total_freight_value',
    'total_items',
    'review_score',
    'max_installments',
    'lat',
    'lng'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. BASIC VALIDATION

print("\n" + "="*70)
print("DATA VALIDATION")
print("="*70)

print("Rows:", len(df))
print("Unique orders:", df['order_id'].nunique())
print("Duplicate orders:", df['order_id'].duplicated().sum())

print("\nOrder Status:")
print(df['order_status'].value_counts())

print("\nMissing values:")
display(
    df.isnull().sum()
      .sort_values(ascending=False)
      .head(15)
)

# 4. DUCKDB CONNECTION


con = duckdb.connect()
con.register("orders", df)

# OVERALL KPIs


total_orders = len(df)
product_revenue = df['total_item_price'].sum()
freight_value = df['total_freight_value'].sum()
total_order_value = df['total_payment_value'].sum()
avg_review = df['review_score'].mean()
late_rate = df['is_late'].mean()

print("\n" + "="*70)
print("OVERALL MARKETPLACE KPIs")
print("="*70)

print(f"Delivered Orders      : {total_orders:,}")
print(f"Product Revenue       : R$ {product_revenue:,.2f}")
print(f"Freight Value         : R$ {freight_value:,.2f}")
print(f"Total Order Value     : R$ {total_order_value:,.2f}")
print(f"Average Review Score  : {avg_review:.2f} / 5")
print(f"Late Delivery Rate    : {late_rate:.2%}")


# QUESTION 1
# MARKETPLACE PERFORMANCE OVER TIME


print("\n\n" + "="*70)
print("QUESTION 1 — MARKETPLACE PERFORMANCE OVER TIME")
print("="*70)

q1_monthly = con.execute("""
SELECT
    DATE_TRUNC('month', order_purchase_timestamp) AS month,
    COUNT(DISTINCT order_id) AS orders,
    SUM(total_item_price) AS product_revenue,
    SUM(total_freight_value) AS freight,
    SUM(total_payment_value) AS total_order_value,
    AVG(review_score) AS avg_review_score
FROM orders
GROUP BY 1
ORDER BY 1
""").df()

print("\nMonthly Performance:")
display(q1_monthly)

peak_month = q1_monthly.loc[q1_monthly['orders'].idxmax()]

print("\nQ1 KEY FINDING")
print("-"*50)
print(
    f"Peak month: {peak_month['month'].strftime('%B %Y')}"
)
print(f"Orders: {peak_month['orders']:,}")
print(f"Product Revenue: R$ {peak_month['product_revenue']:,.2f}")
print(f"Total Order Value: R$ {peak_month['total_order_value']:,.2f}")
print(f"Average Review: {peak_month['avg_review_score']:.2f}")

# Orders chart
plt.figure(figsize=(12,5))
plt.plot(
    q1_monthly['month'],
    q1_monthly['orders'],
    marker='o'
)
plt.title("Monthly Order Volume")
plt.xlabel("Month")
plt.ylabel("Orders")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

# Revenue chart
plt.figure(figsize=(12,5))
plt.plot(
    q1_monthly['month'],
    q1_monthly['product_revenue'],
    marker='o',
    label='Product Revenue'
)
plt.plot(
    q1_monthly['month'],
    q1_monthly['total_order_value'],
    marker='o',
    label='Total Order Value'
)
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (BRL)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()

# Review chart
plt.figure(figsize=(12,5))
plt.plot(
    q1_monthly['month'],
    q1_monthly['avg_review_score'],
    marker='o'
)
plt.axhline(
    avg_review,
    linestyle='--',
    label='Overall Average'
)
plt.title("Monthly Average Review Score")
plt.xlabel("Month")
plt.ylabel("Review Score")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()

print(
    "\nINTERPRETATION:"
    "\nMarketplace volume and revenue increase substantially over time,"
    "\nbut customer satisfaction does not move exactly in line with sales."
)

# QUESTION 2
# DELIVERY PERFORMANCE & CUSTOMER SATISFACTION

print("\n\n" + "="*70)
print("QUESTION 2 — DELIVERY PERFORMANCE & CUSTOMER SATISFACTION")
print("="*70)

# Create delivery buckets
df['delivery_bucket'] = pd.cut(
    df['delivery_delay_days'],
    bins=[-np.inf, 0, 3, 7, np.inf],
    labels=[
        'Early',
        '1–3 days late',
        '4–7 days late',
        '8+ days late'
    ]
)

con.unregister("orders")
con.register("orders", df)

q2_delivery = con.execute("""
SELECT
    delivery_bucket,
    COUNT(*) AS orders,
    AVG(delivery_delay_days) AS avg_delay_days,
    AVG(review_score) AS avg_review_score,
    100.0 *
        SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END)
        / COUNT(*) AS low_rating_pct
FROM orders
GROUP BY delivery_bucket
ORDER BY
    CASE delivery_bucket
        WHEN 'Early' THEN 1
        WHEN '1–3 days late' THEN 2
        WHEN '4–7 days late' THEN 3
        WHEN '8+ days late' THEN 4
    END
""").df()

q2_delivery['avg_delay_days'] = q2_delivery['avg_delay_days'].round(2)
q2_delivery['avg_review_score'] = q2_delivery['avg_review_score'].round(2)
q2_delivery['low_rating_pct'] = q2_delivery['low_rating_pct'].round(2)

print("\nDelivery Performance:")
display(q2_delivery)

# Delivery vs review
plt.figure(figsize=(10,5))
sns.barplot(
    data=q2_delivery,
    x='delivery_bucket',
    y='avg_review_score'
)
plt.title("Delivery Timing vs Average Review Score")
plt.xlabel("Delivery Performance")
plt.ylabel("Average Review Score")
plt.ylim(0,5)
plt.show()

# Low vs high rating
q2_rating_groups = con.execute("""
SELECT
    CASE
        WHEN review_score <= 2 THEN 'Low rating (1–2)'
        WHEN review_score >= 4 THEN 'High rating (4–5)'
        ELSE 'Neutral (3)'
    END AS rating_group,

    COUNT(*) AS orders,

    AVG(delivery_delay_days) AS avg_delay,

    100.0 *
        SUM(CASE WHEN is_late = TRUE THEN 1 ELSE 0 END)
        / COUNT(*) AS late_rate,

    AVG(total_freight_value) AS avg_freight,

    AVG(total_payment_value) AS avg_order_value,

    AVG(review_score) AS avg_review

FROM orders
GROUP BY 1
ORDER BY 1
""").df()

for col in [
    'avg_delay',
    'late_rate',
    'avg_freight',
    'avg_order_value',
    'avg_review'
]:
    q2_rating_groups[col] = q2_rating_groups[col].round(2)

print("\nLow vs High Rating:")
display(q2_rating_groups)

# Correlations
corr_cols = [
    'delivery_delay_days',
    'total_freight_value',
    'total_payment_value',
    'total_item_price',
    'max_installments',
    'review_score'
]

correlations = (
    df[corr_cols]
    .corr()['review_score']
    .drop('review_score')
    .sort_values()
)

print("\nCorrelation with Review Score:")
display(correlations.round(3))

# Calculate 9.6x
low_rating_df = df[df['review_score'] <= 2]
high_rating_df = df[df['review_score'] >= 4]

low_late_rate = low_rating_df['is_late'].mean()
high_late_rate = high_rating_df['is_late'].mean()

late_ratio = low_late_rate / high_late_rate

print("\nQ2 KEY FINDING")
print("-"*50)
print(f"Low-rating late rate : {low_late_rate:.2%}")
print(f"High-rating late rate: {high_late_rate:.2%}")
print(f"Difference           : {late_ratio:.1f}x")

print(
    "\nINTERPRETATION:"
    "\nDelivery delay has the strongest measured relationship with review score."
    "\nAs lateness increases, customer satisfaction falls sharply."
    "\nThis is an ASSOCIATION, not proof of causality."
)


# QUESTION 3
# SELLER & GEOGRAPHIC PATTERNS


print("\n\n" + "="*70)
print("QUESTION 3 — SELLER & GEOGRAPHIC PATTERNS")
print("="*70)

# State analysis
q3_state = con.execute("""
SELECT
    customer_state AS state,
    COUNT(*) AS orders,
    AVG(review_score) AS avg_review,
    AVG(total_freight_value) AS avg_freight,
    AVG(delivery_delay_days) AS avg_delay,

    100.0 *
        SUM(CASE WHEN is_late = TRUE THEN 1 ELSE 0 END)
        / COUNT(*) AS late_rate

FROM orders
GROUP BY customer_state
ORDER BY orders DESC
""").df()

q3_state[
    ['avg_review','avg_freight','avg_delay','late_rate']
] = q3_state[
    ['avg_review','avg_freight','avg_delay','late_rate']
].round(2)

print("\nState-Level Performance:")
display(q3_state)

# Top states
plt.figure(figsize=(10,5))
sns.barplot(
    data=q3_state.head(10),
    x='state',
    y='orders'
)
plt.title("Top States by Order Volume")
plt.xlabel("Customer State")
plt.ylabel("Orders")
plt.show()

# Freight vs review
plt.figure(figsize=(10,6))
sns.scatterplot(
    data=q3_state,
    x='avg_freight',
    y='avg_review',
    size='orders',
    sizes=(50,500)
)
plt.title("Freight Cost vs Customer Satisfaction by State")
plt.xlabel("Average Freight")
plt.ylabel("Average Review")
plt.grid(True)
plt.show()

# Late rate vs review
plt.figure(figsize=(10,6))
sns.scatterplot(
    data=q3_state,
    x='late_rate',
    y='avg_review',
    size='orders',
    sizes=(50,500)
)
plt.title("Late Delivery Rate vs Customer Satisfaction")
plt.xlabel("Late Delivery Rate (%)")
plt.ylabel("Average Review")
plt.grid(True)
plt.show()

print("\nTop States:")
display(q3_state.head(10))

print("\nLowest Review States — minimum 300 orders:")
display(
    q3_state[
        q3_state['orders'] >= 300
    ]
    .sort_values('avg_review')
    .head(10)
)

# Seller analysis
q3_seller = con.execute("""
SELECT
    primary_seller_id AS seller_id,
    COUNT(*) AS orders,
    AVG(review_score) AS avg_review,
    AVG(total_freight_value) AS avg_freight,

    100.0 *
        SUM(CASE WHEN is_late = TRUE THEN 1 ELSE 0 END)
        / COUNT(*) AS late_rate

FROM orders

WHERE primary_seller_id IS NOT NULL

GROUP BY primary_seller_id

HAVING COUNT(*) >= 100

ORDER BY avg_review
""").df()

q3_seller[
    ['avg_review','avg_freight','late_rate']
] = q3_seller[
    ['avg_review','avg_freight','late_rate']
].round(2)

print("\nSeller Screening — minimum 100 orders:")
display(q3_seller.head(15))

print(
    "\nSELLER NOTE:"
    "\nThe cleaned dataset contains primary_seller_id at order level."
    "\nTherefore seller results are a screening analysis,"
    "\nnot perfect multi-seller attribution."
)

print(
    "\nQ3 KEY FINDING:"
    "\nGeography appears to matter mainly through logistics."
    "\nSome lower-volume states have higher freight and/or late rates"
    "\nand lower customer satisfaction."
)


# QUESTION 4
# PRODUCT CATEGORY PERFORMANCE


print("\n\n" + "="*70)
print("QUESTION 4 — PRODUCT CATEGORY PERFORMANCE")
print("="*70)

print(
    "\nThe cleaned CSV does not contain product_id/category information."
)
print(
    "Therefore the ORIGINAL Excel file is required for Q4."
)

print("\nUPLOAD THE ORIGINAL OLIST EXCEL FILE NOW.")
print("It should contain the products, order_items and category_translation sheets.")

uploaded_raw = files.upload()

raw_file = list(uploaded_raw.keys())[0]

print("\nRaw file:", raw_file)

# Read required sheets
order_items = pd.read_excel(
    raw_file,
    sheet_name='order_items'
)

products = pd.read_excel(
    raw_file,
    sheet_name='products'
)

translation = pd.read_excel(
    raw_file,
    sheet_name='category_translation'
)

print("\nRaw tables loaded:")
print("order_items:", order_items.shape)
print("products:", products.shape)
print("translation:", translation.shape)

# Keep only cleaned/delivered orders
clean_order_ids = set(df['order_id'])

order_items_clean = order_items[
    order_items['order_id'].isin(clean_order_ids)
].copy()

print(
    "\nOrder items after restricting to cleaned orders:",
    order_items_clean.shape
)

# Merge product category
category_df = order_items_clean.merge(
    products[
        ['product_id', 'product_category_name']
    ],
    on='product_id',
    how='left'
)

# Merge English translation
category_df = category_df.merge(
    translation,
    on='product_category_name',
    how='left'
)

# Category name
category_df['category'] = (
    category_df['product_category_name_english']
    .fillna(category_df['product_category_name'])
)

# Create order-category level
category_order = category_df.groupby(
    ['order_id', 'category'],
    as_index=False
).agg(
    category_revenue=('price', 'sum'),
    category_freight=('freight_value', 'sum'),
    items=('product_id', 'count')
)

# Add review
category_order = category_order.merge(
    df[
        ['order_id', 'review_score']
    ],
    on='order_id',
    how='left'
)

# Final category table
q4_category = category_order.groupby(
    'category',
    as_index=False
).agg(
    orders=('order_id', 'nunique'),
    items=('items', 'sum'),
    revenue=('category_revenue', 'sum'),
    avg_order_revenue=('category_revenue', 'mean'),
    avg_review=('review_score', 'mean')
)

q4_category['avg_revenue_per_order'] = (
    q4_category['revenue'] /
    q4_category['orders']
)

q4_category = q4_category.sort_values(
    'orders',
    ascending=False
)

q4_category[
    [
        'revenue',
        'avg_order_revenue',
        'avg_review',
        'avg_revenue_per_order'
    ]
] = q4_category[
    [
        'revenue',
        'avg_order_revenue',
        'avg_review',
        'avg_revenue_per_order'
    ]
].round(2)

print("\nTop Product Categories:")
display(q4_category.head(20))

# Category volume
plt.figure(figsize=(12,7))
sns.barplot(
    data=q4_category.head(15),
    y='category',
    x='orders'
)
plt.title("Top Product Categories by Order Volume")
plt.xlabel("Orders")
plt.ylabel("Category")
plt.show()

# Category revenue
top_revenue = (
    q4_category
    .sort_values('revenue', ascending=False)
    .head(15)
)

plt.figure(figsize=(12,7))
sns.barplot(
    data=top_revenue,
    y='category',
    x='revenue'
)
plt.title("Top Product Categories by Revenue")
plt.xlabel("Revenue (BRL)")
plt.ylabel("Category")
plt.show()

# Category review
category_review = (
    q4_category[
        q4_category['orders'] >= 100
    ]
    .sort_values('avg_review')
)

print("\nLowest-rated categories — minimum 100 orders:")
display(category_review.head(15))

plt.figure(figsize=(12,7))
sns.barplot(
    data=category_review.head(15),
    y='category',
    x='avg_review'
)
plt.title("Lowest-Rated Product Categories")
plt.xlabel("Average Review Score")
plt.ylabel("Category")
plt.xlim(0,5)
plt.show()

print(
    "\nQ4 INTERPRETATION:"
    "\nUse this analysis to identify:"
    "\n1. High-volume categories"
    "\n2. High-revenue categories"
    "\n3. High-value categories"
    "\n4. Categories with weaker satisfaction"
    "\n"
    "\nPrioritize categories that combine high demand"
    "\nwith below-average customer satisfaction."
)

# QUESTION 5
# PAYMENT BEHAVIOR


print("\n\n" + "="*70)
print("QUESTION 5 — PAYMENT BEHAVIOR")
print("="*70)

q5_payment = con.execute("""
SELECT
    primary_payment_type AS payment_type,
    COUNT(*) AS orders,
    AVG(total_payment_value) AS avg_order_value,
    AVG(max_installments) AS avg_installments,
    AVG(review_score) AS avg_review

FROM orders

GROUP BY primary_payment_type

ORDER BY orders DESC
""").df()

q5_payment[
    ['avg_order_value','avg_installments','avg_review']
] = q5_payment[
    ['avg_order_value','avg_installments','avg_review']
].round(2)

print("\nPayment Type Analysis:")
display(q5_payment)

# Payment volume
plt.figure(figsize=(9,5))
sns.barplot(
    data=q5_payment,
    x='payment_type',
    y='orders'
)
plt.title("Orders by Payment Type")
plt.xlabel("Payment Type")
plt.ylabel("Orders")
plt.show()

# Payment value
plt.figure(figsize=(9,5))
sns.barplot(
    data=q5_payment,
    x='payment_type',
    y='avg_order_value'
)
plt.title("Average Order Value by Payment Type")
plt.xlabel("Payment Type")
plt.ylabel("Average Order Value (BRL)")
plt.show()

# Installments
q5_installments = con.execute("""
SELECT
    max_installments AS installments,
    COUNT(*) AS orders,
    AVG(total_payment_value) AS avg_order_value,
    AVG(review_score) AS avg_review

FROM orders

WHERE max_installments IS NOT NULL

GROUP BY max_installments

ORDER BY max_installments
""").df()

q5_installments[
    ['avg_order_value','avg_review']
] = q5_installments[
    ['avg_order_value','avg_review']
].round(2)

print("\nInstallment Analysis:")
display(q5_installments)

plt.figure(figsize=(12,5))
sns.barplot(
    data=q5_installments,
    x='installments',
    y='avg_order_value'
)
plt.title("Installments vs Average Order Value")
plt.xlabel("Maximum Installments")
plt.ylabel("Average Order Value (BRL)")
plt.show()

credit_card_orders = (
    df['primary_payment_type']
    .eq('credit_card')
    .sum()
)

credit_card_share = credit_card_orders / len(df)

print("\nQ5 KEY FINDING")
print("-"*50)
print(f"Credit Card Orders: {credit_card_orders:,}")
print(f"Credit Card Share : {credit_card_share:.2%}")

print(
    "\nINTERPRETATION:"
    "\nCredit card is the dominant payment method and has the highest"
    "\naverage order value, while review scores vary only slightly"
    "\nacross payment types."
)

# QUESTION 6
# ROOT CAUSE ANALYSIS

print("\n\n" + "="*70)
print("QUESTION 6 — ROOT CAUSE ANALYSIS")
print("="*70)

q6 = con.execute("""
SELECT

    CASE
        WHEN review_score <= 2 THEN 'Low rating (1–2)'
        WHEN review_score >= 4 THEN 'High rating (4–5)'
        ELSE 'Neutral (3)'
    END AS rating_group,

    COUNT(*) AS orders,

    AVG(review_score) AS avg_review,

    AVG(delivery_delay_days) AS avg_delay,

    100.0 *
        SUM(CASE WHEN is_late = TRUE THEN 1 ELSE 0 END)
        / COUNT(*) AS late_rate,

    AVG(total_freight_value) AS avg_freight,

    AVG(total_payment_value) AS avg_order_value,

    AVG(max_installments) AS avg_installments,

    100.0 *
        AVG(
            CASE
                WHEN has_comment = TRUE THEN 1
                ELSE 0
            END
        ) AS comment_rate

FROM orders

GROUP BY 1

ORDER BY 1
""").df()

q6[
    [
        'avg_review',
        'avg_delay',
        'late_rate',
        'avg_freight',
        'avg_order_value',
        'avg_installments',
        'comment_rate'
    ]
] = q6[
    [
        'avg_review',
        'avg_delay',
        'late_rate',
        'avg_freight',
        'avg_order_value',
        'avg_installments',
        'comment_rate'
    ]
].round(2)

print("\nLow vs High Satisfaction:")
display(q6)

# Correlation matrix
driver_cols = [
    'delivery_delay_days',
    'total_freight_value',
    'total_payment_value',
    'total_item_price',
    'max_installments',
    'total_items',
    'review_score'
]

corr_matrix = df[driver_cols].corr()

plt.figure(figsize=(9,7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f'
)
plt.title("Correlation Between Review Score and Key Factors")
plt.show()

driver_corr = (
    corr_matrix['review_score']
    .drop('review_score')
    .sort_values()
)

print("\nCorrelation with Review Score:")
display(driver_corr.round(3))

# ROOT CAUSE COMPARISON


low_rating = df[df['review_score'] <= 2]
high_rating = df[df['review_score'] >= 4]

low_late = low_rating['is_late'].mean()
high_late = high_rating['is_late'].mean()

ratio = low_late / high_late

print("\n" + "="*70)
print("ROOT CAUSE FINDINGS")
print("="*70)

print(
    f"\nLow-rating customers late rate : {low_late:.2%}"
)

print(
    f"High-rating customers late rate: {high_late:.2%}"
)

print(
    f"\nLow-rated customers are approximately {ratio:.1f}x"
    " more likely to experience a late delivery."
)

print(
    "\nPRIMARY CONTRIBUTOR:"
    "\nDelivery reliability / delivery delay"
)

print(
    "\nSECONDARY CONTRIBUTOR:"
    "\nFreight / logistics cost"
)

print(
    "\nWEAKER CONTRIBUTORS:"
    "\nOrder value, product value and installments"
)

print(
    "\nDO NOT CALL THIS A CAUSE:"
    "\nComment presence."
    "\nLow-rated customers comment more frequently, but this is"
    "\ncustomer response behavior rather than an operational root cause."
)


# FINAL EXECUTIVE SUMMARY


print("\n\n" + "="*70)
print("FINAL EXECUTIVE SUMMARY")
print("="*70)

print(f"""
1. MARKETPLACE PERFORMANCE
   • {total_orders:,} delivered orders analyzed.
   • Product revenue: R$ {product_revenue:,.2f}
   • Total order value: R$ {total_order_value:,.2f}
   • Peak month: {peak_month['month'].strftime('%B %Y')}
   • Peak-month orders: {peak_month['orders']:,}

2. CUSTOMER SATISFACTION
   • Overall average review: {avg_review:.2f}/5
   • Late delivery rate: {late_rate:.2%}
   • Delivery delay is the strongest measured factor associated
     with customer satisfaction.

3. DELIVERY
   • Early deliveries average roughly 4.28/5.
   • 4–7 days late drops to roughly 2.30/5.
   • 8+ days late drops to roughly 1.72/5.
   • Low-rated customers are about {ratio:.1f}x more likely
     to experience a late delivery.

4. GEOGRAPHY
   • São Paulo is the dominant market by order volume.
   • Geographic differences are associated with differences
     in freight cost, late delivery and review scores.

5. PRODUCT CATEGORY
   • Category performance must be evaluated using the original
     product tables because category information is absent
     from the cleaned master dataset.
   • High-volume + low-review categories should be prioritized.

6. PAYMENT
   • Credit card is the dominant payment method.
   • Credit card orders also have the highest average order value.
   • Payment method shows relatively little variation in review scores.

7. ROOT CAUSE
   • PRIMARY: Delivery reliability
   • SECONDARY: Freight/logistics
   • WEAKER: Order value, product value and installments

OVERALL BUSINESS CONCLUSION:
The strongest opportunity for Olist is improving delivery reliability,
especially in regions where logistics costs and late-delivery rates
are relatively high. Increasing marketplace volume alone is not enough;
the customer experience must scale with it.
""")


# EXPORT ALL RESULTS


print("\n" + "="*70)
print("EXPORTING RESULTS")
print("="*70)

output_file = "Olist_Hackathon_Complete_Analysis.xlsx"

with pd.ExcelWriter(
    output_file,
    engine='openpyxl'
) as writer:

    q1_monthly.to_excel(
        writer,
        sheet_name='Q1_Monthly',
        index=False
    )

    q2_delivery.to_excel(
        writer,
        sheet_name='Q2_Delivery',
        index=False
    )

    q2_rating_groups.to_excel(
        writer,
        sheet_name='Q2_Rating_Groups',
        index=False
    )

    q3_state.to_excel(
        writer,
        sheet_name='Q3_State',
        index=False
    )

    q3_seller.to_excel(
        writer,
        sheet_name='Q3_Seller',
        index=False
    )

    q5_payment.to_excel(
        writer,
        sheet_name='Q5_Payment',
        index=False
    )

    q5_installments.to_excel(
        writer,
        sheet_name='Q5_Installments',
        index=False
    )

    q6.to_excel(
        writer,
        sheet_name='Q6_Root_Cause',
        index=False
    )

    corr_matrix.to_excel(
        writer,
        sheet_name='Correlations'
    )

    if 'q4_category' in globals():
        q4_category.to_excel(
            writer,
            sheet_name='Q4_Category',
            index=False
        )

print("\nAnalysis complete!")
print("Excel file created:", output_file)

files.download(output_file)

print("\n" + "="*70)
print("DONE")
print("="*70)

File found at: /Document from singhlink4.xlsx
Master and summary datasets processed successfully! Triggering downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')